# Investigating the vendor software

In the previous practical, we obtained a reading and started exploring its bytes. Now we have another source of clues: the software that controls the reader.

Our main task is to finish the luminescence reading: decode all 96 values, establish their order, and complete a Python function that reads them. If you have time, investigate absorbance afterward.

The C# code is provided in the **handout folder**. Open it in any text editor, such as Notepad, alongside this notebook. You do not need VS Code, a decompiler, or anything that runs C#.

Keep [captures/reading.txt](captures/reading.txt) beside the notebook. Use your local Python kernel, pySerial, and the Pico's known serial port for the hardware sections. Close other programs using that port first.

The Pico's onboard LED is on when the simulated door is open and off when it is closed.


## 1. A needle in a haystack

Last time, we received a long reply and tried to turn its bytes into measurements. We could compare a few numbers with the transcript, but some questions remained: where exactly did the measurement data start, and was the application doing anything else to those values?

The vendor application already knows how to display a reading. Somewhere in its code is the conversion we need—but finding it can feel like looking for a needle in a haystack.

Take a look through the handout folder before opening a file. There is code for settings, reports, screen updates, and other application features. Most of it will not help us decode a measurement. In a larger vendor application, the few lines we need might be buried among thousands of files. Having the code does not immediately tell us where the answer is.

We do not need to understand the whole application. We need a way into the relevant part. A familiar screen message is a useful clue: **“Reading plate...”**. Somewhere, the application displays that message just before doing the work we care about.

For this first investigation, the file paths below give us a starting point. As we follow them, notice what leads from one place to the next: a screen message leads to a measurement call, and that call leads to the byte-decoding code. This is the search technique we can reuse when nobody has pointed out the files for us.

In the handout folder, open `PlateReader.Client.Application/MeasurementCoordinator.cs` in Notepad. Use **Find** (Ctrl+F) for `Reading plate...`. This takes us into `Run`, the method that organizes a measurement. Nearby you will see calls to `ReadLuminescence` and `ReadAbsorbance`: the application chooses one according to the selected mode.

We are finishing luminescence first, so let's follow `ReadLuminescence`. Its definition is in `PlateReader.Client.Device/PlateReaderDevice.cs`; search for `public PlateResult ReadLuminescence`.

Here is that method from the supplied code:

```csharp
public PlateResult ReadLuminescence(string sampleName)
{
    ushort[] array = PacketCodec.ReadArray(connection.Exchange(Command.ReadLuminescence, 0), 4, 96);
    double[] array2 = new double[array.Length];
    for (int i = 0; i < array.Length; i++)
    {
        array2[i] = (int)array[i];
    }
    return new PlateResult(sampleName, MeasurementMode.Luminescence, "RLU", array2);
}
```

A C# method is much like a Python function. You can recognize a call by its name followed by parentheses. Here, look for the call to `ReadArray`. The numbers passed to it are clues about which part of the reply contains our measurements. We have narrowed the search from the whole application to one method and the helper it calls. The reporting and screen code can wait; this is the part that can answer our current question.

All file paths in this notebook start from the handout folder. The files are plain text, so Notepad is enough to follow along.

## 2. How does the application turn bytes into numbers?

In `ReadLuminescence`, the application receives a reply and passes it to `ReadArray`. Before copying that into Python, we need to understand what this helper does.

Keep `PlateReader.Client.Device/PlateReaderDevice.cs` open. In another window, open `PlateReader.Client.Device/PacketCodec.cs` and search for `public static ushort[] ReadArray`.

```csharp
public static ushort[] ReadArray(byte[] buffer, int offset, int count)
{
    ushort[] array = new ushort[count];
    for (int i = 0; i < count; i++)
    {
        array[i] = ReadUInt16(buffer, offset + i * 2);
    }
    return array;
}
```

The names of its parameters help us read the call: `buffer` is the received data, `offset` is where to start, and `count` is how many values to read. Compare these with the arguments in `ReadLuminescence`. You can now see where the application starts reading and how many measurements it expects.

Inside the loop, `ReadUInt16` converts two bytes into one whole number. Use `struct.unpack(">H", pair)` for the same conversion in Python. The `i * 2` advances by two bytes for each value.

A few pieces of C# will help:

| C# | Meaning |
|---|---|
| `byte[]` | An array of bytes, used here for the reply |
| `ushort` | A whole number from 0 to 65535, stored in two bytes |
| `ushort[]` | An array of those whole numbers |
| `double[]` | An array of floating-point numbers |
| `values[i]` | The value at index `i`, like indexing a Python list |

Finally, look back at `ReadLuminescence` in `PlateReader.Client.Device/PlateReaderDevice.cs`. Follow a value from `ReadArray` into the returned result. Does the application change its magnitude, or just store it in a different type? This tells us whether unpacking the integer is enough.

### Worked example: bytes to readings

Run the next cell as supplied. It needs no Pico connection or earlier cells.

The six bytes `02 B2 04 49 02 16` come from `reading.txt`. Each pair represents one reading. Use the supplied `>H` format to read each two-byte whole number, matching the vendor code.

The loop takes two bytes at a time, converts them, and adds the number to `decoded`. It prints each conversion, then compares all three numbers with the vendor display.

In [1]:
import struct

# Three readings copied from the capture.
example_bytes = bytes.fromhex("02 B2 04 49 02 16")
displayed_values = [690, 1097, 534]

# Decode each pair and collect the readings.
decoded = []
for offset in range(0, len(example_bytes), 2):
    pair = example_bytes[offset:offset + 2]
    value, = struct.unpack(">H", pair)
    decoded.append(value)
    print(pair.hex(" ").upper(), "->", value)

# Compare our numbers with the values shown by the vendor application.
print("Decoded:", decoded)
print("Displayed:", displayed_values)
assert decoded == displayed_values

02 B2 -> 690
04 49 -> 1097
02 16 -> 534
Decoded: [690, 1097, 534]
Displayed: [690, 1097, 534]


## 3. Make the decoder reusable

The same conversion needs to happen for every well. We can put it in `decode_luminescence(reply)`, so the rest of our program can ask for measurements without knowing how the bytes are arranged.

The supplied line `data_bytes = reply[4:-2]` removes the four header bytes and the final two bytes. This leaves the 192 measurement bytes: 96 pairs.

Use the completed loop above as your example. Loop over `data_bytes` in steps of two, unpack each pair, and append its number to `values`. The function already returns that list.

The checks below are already filled in. Remember the short reply from the previous practical: we do not want to treat an error message as measurements. These checks make us stop if the reply is not the complete successful reading we expect.

You can see the same idea in `PlateReader.Client.Device/PacketCodec.cs`: search for `public static int GetResponseLength` and `public static void ValidateResponse`. We will concentrate on the conversion here.

In [ ]:
def decode_luminescence(reply):
    assert len(reply) == 198, "Expected a complete luminescence reply."
    assert reply[:4] == bytes.fromhex("A7 83 20 00"), "Unexpected header or status."
    assert reply[-2:] == bytes.fromhex("00 FF"), "Unexpected ending."
    data_bytes = reply[4:-2]  # Remove the header and ending bytes.
    values = []
    # Loop over data_bytes in pairs, unpack each pair, and append its value.
    return values

## 4. Use it with the reader

So far, we have tried a few saved bytes. Let's use the decoder with a fresh reading from the Pico.

We already learned the preparation sequence in the previous practicals: close the door, then read. We can reuse those functions and add our new decoder to `read_luminescence()`.

Disconnect the previous notebook with `pico.close()` first, so this one can use the serial port. Then enter your port below and connect. We do not need to power-cycle this time: we want the reader ready to measure.

In [ ]:
import serial
import time

PORT = "YOUR_PORT"
pico = serial.Serial(PORT, baudrate=115200, timeout=1, write_timeout=1)
time.sleep(1)
pico.reset_input_buffer()

Here is `close_door()` using the exchange we tested earlier. If you have your own version, you can copy it here instead.

Each function keeps a command together with its reply. That lets us prepare the reader with two simple calls at the bottom of the cell. Remember that `close_door()` moves the door, while `pico.close()` releases the serial connection.

In [ ]:
def close_door():
    request = bytes.fromhex("A7 03 07 00 00 00 00 FF")
    pico.write(request)
    reply = pico.read(8)
    assert reply == bytes.fromhex("A7 83 07 00 00 00 00 FF"), "Unexpected Close reply."
    return reply


close_door()

Our reading function already knows how to send the request and receive bytes. Now we can give those bytes meaning.

The familiar exchange is filled in below. Add a call to `decode_luminescence(reply)` and return its result. Then try `readings = read_luminescence()`: `readings` should now contain the measurement values we wanted, ready for the rest of our program to use.

In [ ]:
def read_luminescence():
    request = bytes.fromhex("A7 03 20 00 00 00 00 FF")
    pico.write(request)
    reply = pico.read(198)
    # Decode reply and return the values.


# Uncomment after adding the decoder:
# readings = read_luminescence()
# print("First twelve:", readings[:12])
# assert len(readings) == 96

With matching settings, the readings match the transcript exactly. Gain can persist from an earlier session, so check it before comparing.

If a length or header check fails, inspect the reply before continuing. As we saw earlier, a failed measurement can return eight bytes. Asking `read()` for more does not make the device send more.

## 5. Put the values in their wells

We have 96 readings in a flat list. A plate has 8 rows, with 12 readings in each row.

Open `PlateReader.Client.Domain/PlateResult.cs` in the handout and find `ToRows()`. This method builds the plate one row at a time. The desktop application uses it to arrange the results.

The C# code does the same things we can do with Python lists:

| C# | Python meaning |
| --- | --- |
| `new List<double>()` | Make an empty list: `[]` |
| `list2.Add(values[num])` | Add the next reading: `row.append(readings[index])` |
| `num++` | Move to the next reading: `index += 1` |
| `list.Add(list2)` | Add the finished row: `plate.append(row)` |

The outer loop makes 8 rows. The inner loop puts 12 readings into the current row. The reading index keeps going when a new row starts: the first row uses readings 0–11, then the second starts at reading 12.

Follow that same sequence below. **Fill in the two `append()` calls**: first add `readings[index]` to `row`, then add the finished `row` to `plate`. The loops and index update are supplied.

In [ ]:
# Uncomment after filling in the two append() calls.
# plate = []
# index = 0
# for row_number in range(8):
#     row = []
#     for column_number in range(12):
#         row.append(...)  # Add readings[index].
#         index += 1
#     plate.append(...)  # Add the finished row.
#
# for label, row in zip("ABCDEFGH", plate):
#     print(label, row)

Now `read_luminescence()` gives us measurements, and the row loop puts them in their wells. We have joined the pieces from the earlier practicals into a usable reading.

Try taking another reading and arranging it into rows. Keep the first set so you can compare them. The workshop reader uses a fixed synthetic plate: repeated measurements with the same gain return identical luminescence values. If they differ, check the gain and the bytes received before investigating the decoder.

This completes the main task. If you have time, the other measurement mode gives us something new to investigate below. Otherwise, use the **Disconnect** cell at the end when you are finished.

## 6. If you have time: investigate absorbance

Luminescence gave us one integer per well. The application also offers absorbance, so it is tempting to reuse the same decoder. Let's see whether its code supports that idea.

Keep [captures/absorbance.txt](captures/absorbance.txt) open beside the C# code. It shows the setup exchanges, a reading, and the values displayed for that reading. We can use this to check our interpretation of the code against the bytes.

### 6.1. Follow one result through the calculation

Open `PlateReader.Client.Device/PlateReaderDevice.cs` and find `public PlateResult ReadAbsorbance`. Compare it with `ReadLuminescence` above it.

There is more than one `ReadArray` call this time. Look at where each starts and how many values it reads. Those offsets tell us how the data is arranged in the reply.

Then follow the loop for just one well: which values go into the calculation, and what comes out? The conversion to `double` matters in C#, where dividing two integers would discard the fractional part. Python's `/` already performs decimal division.

Notice the check just before that calculation, too. It tells us about a value that would make the calculation fail.

Before trying this on the Pico, we also need to know how much data to receive. In `PlateReader.Client.Device/PacketCodec.cs`, find `public static int GetResponseLength` and look for the absorbance case. Does its length fit the data regions you found?

### 6.2. Prepare an absorbance measurement

In the previous practical, copying the final read request was not enough: an earlier command made the difference. We should check the setup sequence for this mode too.

Return to `PlateReader.Client.Application/MeasurementCoordinator.cs` and find `Run(`. Look above the measurement call. The application sets gain and, for absorbance, wavelength. We implemented `set_gain()` from traffic in the reading practical. Now we can check that command against the vendor code and add wavelength control. In `absorbance.txt`, find the comment about selecting absorbance, wavelength 450 nm, and gain 1. Compare those requests with the commands and arguments you find in the C# code.

In `PlateReader.Client.Device/PlateReaderDevice.cs`, find `public void SetWavelength`, then `public void SetGain`. Each method checks the parameter and passes it to `Exchange` together with a command name. The allowed ranges are useful when we try our own settings.

Those names still need to become bytes. Open `PlateReader.Client.Device/Command.cs` and find `enum Command`: this associates each command name with a number. The command numbers may be written in decimal (as a result of decompilation), while our transcript uses hex. For example, decimal 167 is `0xA7`; Python's `hex(...)` can help compare them.

Finally, open `PlateReader.Client.Device/PacketCodec.cs` and find `public static byte[] BuildCommand`. Recognize the fixed bytes from our earlier requests? Look at where it puts the command and the two parts of the argument.

This reveals something about our earlier gain command: the `00` we treated as fixed is actually the high byte of a two-byte argument. Our gains were only 1–16, so that byte was always zero and changing the low byte worked. The vendor code shows the full field: bytes 4 and 5, high byte first.

Wavelength makes both bytes useful. For example, 450 is `01 C2`, which does not fit in one byte. `struct.pack(">H", argument)` builds both bytes for us. For gain 2 it produces `00 02`, giving exactly the same command as our earlier construction.

We can build the same message in Python. Complete `build_command()` below, using `struct.pack()` for the argument. As before, creating bytes does not send them—we can print and check them first.

In [ ]:
def build_command(command, argument=0):
    # Combine the fixed bytes, command, packed argument, and final bytes.
    pass

# Check against a message we already observed:
# assert build_command(0x07) == bytes.fromhex("A7 03 07 00 00 00 00 FF")
# print(build_command(..., 450).hex(" "))  # Wavelength
# print(build_command(..., 1).hex(" "))    # Gain

### Give the settings useful names

Reuse the `set_gain()` function from the reading practical, or implement it below using `build_command()`. Add `set_wavelength()` in the same way. This notebook runs independently, so include both definitions here.

If using the skeletons below, complete each with a request from `build_command()`, a write, and an eight-byte read. The reply repeats the accepted parameter, which gives us a way to check that the reader accepted our setting.

For the vendor's version of that check, look at `public void SetWavelength` and `public void SetGain` in `PlateReader.Client.Device/PlateReaderDevice.cs`. Include a check for a complete successful reply before inspecting its parameter.

Then try `set_wavelength(450)` and `set_gain(1)`. These settings give us a starting point for the first absorbance reading.

In [ ]:
def set_wavelength(wavelength):
    # Build the wavelength request and send it.
    # Read and check its eight-byte reply, then return it.
    pass


def set_gain(gain):
    # Build the gain request and send it.
    # Read and check its eight-byte reply, then return it.
    pass


# set_wavelength(450)
# set_gain(1)

### 6.3. Translate the absorbance decoder

We can reuse the byte-conversion technique from luminescence, but now we need to keep the two data regions separate before combining their values.

Build two lists with short `struct.unpack()` loops. Use the offsets from `ReadAbsorbance` in `PlateReader.Client.Device/PlateReaderDevice.cs` to select the right pairs. Then work through the well indices and apply the calculation you found to the corresponding values.

Check the denominator before dividing, as the vendor code does. The header and length checks are already supplied so you can focus on turning the data into results.

In [ ]:
def decode_absorbance(reply):
    assert len(reply) == 390, "Expected a complete absorbance reply."
    assert reply[:4] == bytes.fromhex("A7 83 21 00"), "Unexpected header or status."
    assert reply[-2:] == bytes.fromhex("00 FF"), "Unexpected ending."
    # Unpack both arrays, then calculate one result for each well.
    pass

### Try the decoder on the captured reading

Before taking a new measurement, we have one where we know what the application displayed. In `captures/absorbance.txt`, find `# Action: user clicks Read`. Copy the complete `<` line immediately below its request into the cell below, leaving out the `<` label.

Run that reply through your decoder and compare the first three results with row A at the bottom of the file. The display rounds to four decimal places, so round your values for this comparison too. With the same settings, the Pico returns these same values.

If the values disagree, go back to the two array offsets and the calculation in the C# method. This gives us a concrete way to check our translation before adding a live exchange.

In [ ]:
captured_reply = bytes.fromhex("")  # Copy the complete absorbance < line here.
assert len(captured_reply) == 390, "Copy the complete reply before decoding."
captured_values = decode_absorbance(captured_reply)
print("First three:")
for value in captured_values[:3]:
    print(round(value, 4))

With the settings in place, we can try our decoder on a real reply. Fill in the absorbance command and response length you found in the C# code, then uncomment the cell. The printed header and length let us check what arrived before looking at the decoded values.

In [ ]:
# pico.write(build_command(...))
# absorbance_reply = pico.read(...)
# print("Bytes received:", len(absorbance_reply))
# print("Header:", absorbance_reply[:4].hex(" "))
# absorbance = decode_absorbance(absorbance_reply)
# assert len(absorbance) == 96
# print("First twelve:", absorbance[:12])

### Check one well independently

Take the first two bytes of each data region from the **same saved reply**. Unpack those two numbers separately and calculate the expected first result. Compare it with `absorbance[0]`.

This catches mistakes such as selecting the wrong array boundary or reversing the calculation. A list of 96 plausible numbers alone would not catch those errors.

In [ ]:
# first_value, = struct.unpack(..., absorbance_reply[...:...])
# second_value, = struct.unpack(..., absorbance_reply[...:...])
# expected = ...
# print(first_value, second_value, expected, absorbance[0])
# assert absorbance[0] == expected

### 6.4. Bring the steps together

We have now tried the individual exchanges. Wrapping them in `read_absorbance(wavelength=450, gain=1)` will let the caller ask for a measurement in one line.

Use the functions we already have: `close_door()`, `set_wavelength(wavelength)`, and `set_gain(gain)`. Follow them with the read exchange and return the decoded values. Check the parameter ranges before starting, so an invalid setting does not leave us partway through preparation.

Once it works, try 450 nm and 600 nm with gain held at 1. Compare a few wells. Repeating a measurement with the same wavelength and gain returns identical values. Compare those repeated readings with the values after changing wavelength to see the effect of that setting.

In [ ]:
def read_absorbance(wavelength=450, gain=1):
    # Reuse the exchanges and decoder above.
    pass

# values_450 = read_absorbance(450)
# values_600 = read_absorbance(600)
# print(values_450[:3])
# print(values_600[:3])

## 7. Disconnect

Run the cell below when you are finished to release the serial port.

You can now use the vendor code to turn the reply into luminescence values and place them in the plate. If you tried absorbance, you also found how the application calculates a result from two transmitted values.

If the code disagrees with your reader, test the behavior on the device. Software versions can differ.

In [ ]:
pico.close()
print("Serial connection closed.")